# TGEN-TBP-PacBioReseq-GubbinsDetectedEvents - Processing Gubbins outputs

# import statements

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
import os

In [3]:
import glob

In [4]:
# https://bioframe.readthedocs.io/en/latest/guide-intervalops.html
import bioframe as bf


In [5]:
import json


#### Set matplotlib text export settings for Adobe Illustrator

In [6]:
import matplotlib
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

#### Pandas Viewing Settings

In [7]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# Import/parse processed H37rv genome annotations

In [8]:
RepoRef_Dir = "../../References"

AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir = f"{RepoRef_Dir}/201027_H37rv_AnnotatedGenes_And_IntergenicRegions"
H37Rv_GenomeAnnotations_Genes_TSV = f"{AnnotatedGenes_And_IntergenicRegions_RepoRef_Dir}/H37Rv_GenomeAnnotations.Genes.tsv"

## H37Rv Gene Annotations TSV
H37Rv_GenomeAnno_Genes_DF = pd.read_csv(H37Rv_GenomeAnnotations_Genes_TSV, sep = "\t")
H37Rv_GeneInfo_Subset_DF = H37Rv_GenomeAnno_Genes_DF[["H37rv_GeneID", "Symbol", "Feature", "Functional_Category", "Is_Pseudogene", "Product", "PEandPPE_Subfamily", "ExcludedGroup_Category"]]

RvID_To_Symbol_Dict = dict(H37Rv_GeneInfo_Subset_DF[['H37rv_GeneID', 'Symbol']].values)

ESX_Genes_List_TSV = f"{RepoRef_Dir}/190927_H37rv_ListOf_ESXgenes.tsv"
Esx_Genes_DF = pd.read_csv(ESX_Genes_List_TSV, sep = '\t')

In [9]:
H37Rv_GenomeAnno_Genes_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category
0,NC_000962.3,0,1524,+,Rv0001,dnaA,CDS,information pathways,No,Chromosomal replication initiator protein DnaA,NaN,NotExcluded


# Parse in homology H37Rv mapping results (k19w19)

In [10]:
Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8"

H37_Rv_MM2_HomologyMapping_Dir = f"{Main_Project_Dir}/220502.H37Rv.HomologyMapping.k19w19.ProcessedData"       

# Define paths to output TSVs

### Homologous regions (MERGED)
H37Rv_HomologyRegions_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologousRegions.k19w19.tsv"

### Homology map (pairwise alignments)
H37Rv_HomologyMap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.tsv"
H37Rv_HomologyMap_NoOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.NoOverlap.tsv"
H37Rv_HomologyMap_OnlyOverlap_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.OnlyOverlap.tsv"

H37Rv_HomologyMap_NoOverlap_Processed_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HmMap.k19w19.NoOverlap.Processed.V2.tsv"

### Variants from the homology map alignments
H37Rv_HmMap_Var_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.tsv"
H37Rv_HmMap_Var_SNPs_TSV = f"{H37_Rv_MM2_HomologyMapping_Dir}/H37rv.HomologyMap.k19w19.variants.snps.tsv"

### Parse in labeled homology-regions (w/ unique IDs)

HM_MergedRegions_Anno_DF = pd.read_csv(H37Rv_HomologyRegions_TSV, sep="\t")


### Parse in homology-map DFs (pairwise alignments between all homologous regions)

Mtb_HM_PAF_DF = pd.read_csv(H37Rv_HomologyMap_TSV, sep="\t")
Mtb_HM_PAF_DF_NoOverlapRegions = pd.read_csv(H37Rv_HomologyMap_NoOverlap_TSV, sep="\t")
Mtb_HM_PAF_DF_OnlyOverlapRegions = pd.read_csv(H37Rv_HomologyMap_OnlyOverlap_TSV, sep="\t")

HmPair_DF = pd.read_csv(H37Rv_HomologyMap_NoOverlap_Processed_TSV, sep="\t")

# Parse `TGEN-937-SR` sample metadata

In [11]:
Repo_MainDir = "../.."

Repo_DataDir = f"{Repo_MainDir}/Data"

Repo_RunInfoDir = f"{Repo_MainDir}/runInfo_TSVs"

TGen936SR_SampleInfo_CSV_PATH = f"{Repo_RunInfoDir}/TBportals.allTGEN_srrIds.forMax.csv"

TGen_1K_SM_V1_ResultsSummary_Dir = f"{Repo_DataDir}/Tgen1K_WGS_RunMetadata/211019_SM_TGen_1K_V1_ResultsSummary"

TGen936_SampleInfo_Filt_TSV_PATH = f"{Repo_DataDir}/Tgen1K_WGS_RunMetadata/211019_SM_TGen_1K_SampleInfo_V1.F2andCovFiltered.tsv"


TGenSR_WGS_Stats_Filt_DF = pd.read_csv(TGen936_SampleInfo_Filt_TSV_PATH, sep = "\t")

TGenSR_WGS_Stats_Filt_DF["PrimaryLineage"] = TGenSR_WGS_Stats_Filt_DF["PrimaryLineage_Ill"]
TGenSR_WGS_Stats_Filt_DF["SampleID"] = TGenSR_WGS_Stats_Filt_DF["SampleName"]
TGenSR_WGS_Stats_Filt_DF["Lineage"] = TGenSR_WGS_Stats_Filt_DF["LineageCall_Illumina"]
TGenSR_WGS_Stats_Filt_DF.shape

(937, 10)

### Define dictionaries that map sampleID to metadata labels

In [12]:
TGENSR_ID_To_PrimLineage_Dict = dict( TGenSR_WGS_Stats_Filt_DF[['SampleID', 'PrimaryLineage']].values)
TGENSR_ID_To_SubLineage_Dict  = dict( TGenSR_WGS_Stats_Filt_DF[["SampleID", "Lineage"]].values)
TGENSR_ID_To_Dataset_Dict     = dict( TGenSR_WGS_Stats_Filt_DF[['SampleID', 'Dataset_Tag']].values) 


# Define `TBP22-22CI` dataset metadata (Resequenced w/ PacBio HiFi)

### Define explicit lists of isolateIDs + a mapping dictionary of eventIDs to isolateIDs

In [13]:

TGENSR_SelectedIsolates_GCEventVerf = ['TB8073', 'TB6755', 'TB6973', 'TB6778', 'TB3572', 'TB6786', 'TB6552', 'TB6599',
                                       'TB6765', 'TB6977', 'TB6733', 'TB3898', 'TB3706', 'TB3256', 'TB3305', 'TB6976',
                                       'TB6807', 'TB4414', 'TB7340', 'TB6846', 'TB6596', 'TB7044']

dictOf_TGEN_GCEventVerf_TBP_ID_Mappings = {
    "Event_001": {"Target": "TB6733", "Control": "TB3898"},
    "Event_003": {"Target": "TB6599", "Control": "TB6977"},
    "Event_006": {"Target": "TB3305", "Control": "TB3706"},
    "Event_007": {"Target": "TB6755", "Control": "TB7044"},
    "Event_010": {"Target": "TB6552", "Control": "TB6765"},
    "Event_011": {"Target": "TB6778", "Control": "TB6973"},
    "Event_013": {"Target": "TB6786", "Control": "TB3256"},
    "Event_019": {"Target": "TB6977", "Control": "TB6976"},
    "Event_021": {"Target": "TB3572", "Control": "TB6976"},
    "Event_022": {"Target": "TB6596", "Control": "TB8073"},
    "Event_024": {"Target": "TB7340", "Control": "TB6807"},
    "Event_025": {"Target": "TB6846", "Control": "TB4414"},
}


TGEN937SR_ReseqEventIDs = list( dictOf_TGEN_GCEventVerf_TBP_ID_Mappings.keys())
TGEN937SR_ReseqEventIDs


['Event_001',
 'Event_003',
 'Event_006',
 'Event_007',
 'Event_010',
 'Event_011',
 'Event_013',
 'Event_019',
 'Event_021',
 'Event_022',
 'Event_024',
 'Event_025']

In [14]:
TGEN937SR_ReseqEventIDs

['Event_001',
 'Event_003',
 'Event_006',
 'Event_007',
 'Event_010',
 'Event_011',
 'Event_013',
 'Event_019',
 'Event_021',
 'Event_022',
 'Event_024',
 'Event_025']

### Define file paths to TSVs

In [15]:
Repo_MainDir = "../.."
Repo_DataDir = f"{Repo_MainDir}/Data"

TBP22_GCEVerf_Metadata_Dir = f"{Repo_DataDir}/TBP22.22CI.GCEVerfIsolates.Metadata" 

TBP22_22CI_HybridAsmQCStats_TSV_PATH                = f"{TBP22_GCEVerf_Metadata_Dir}//250801.TBP22.22CI.GCEVerfIsolates.HybridAsmQCStats.V1.tsv"

TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.GCEVerfIsolates.Asm_LR_SR.InputPATHs.tsv"

TBP22_GCEvent_To_IsolateIDs_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.TGENSR_Reseq.GCEvent_To_IsolateIDs.tsv"

TBP22_IsolateID_To_GCEvents_TSV = f"{TBP22_GCEVerf_Metadata_Dir}/250801.TBP22.22CI.TGENSR_Reseq.IsolateID_To_GCEvents.tsv"


### Read in DFs defining event to isolateID mappings

In [16]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF = pd.read_csv(TBP22_GCEvent_To_IsolateIDs_TSV, sep="\t" )

# Create EventID → Target dictionary
EventID_to_TargetIsolateID = TBP22_Reseq_GCEvent_To_IsolateIDs_DF.set_index("EventID")["Verification_IsolateID"].to_dict()

# Create EventID → Control dictionary
EventID_to_ControlIsolateID = TBP22_Reseq_GCEvent_To_IsolateIDs_DF.set_index("EventID")["Control_IsolateID"].to_dict()


TBP22_Reseq_GCEvent_To_IsolateIDs_DF.shape

(12, 5)

In [17]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
0,Event_001,TB6733,TB3898,SRR10379945,SRR10397263
1,Event_003,TB6599,TB6977,SRR10379958,SRR10380218
2,Event_006,TB3305,TB3706,SRR10397175,SRR10397096
3,Event_007,TB6755,TB7044,SRR10379935,SRR10380192
4,Event_010,TB6552,TB6765,SRR10380108,SRR10379924
5,Event_011,TB6778,TB6973,SRR10380252,SRR10380223
6,Event_013,TB6786,TB3256,SRR10380244,SRR10397205
7,Event_019,TB6977,TB6976,SRR10380218,SRR10380219
8,Event_021,TB3572,TB6976,SRR10397163,SRR10380219
9,Event_022,TB6596,TB8073,SRR10379961,SRR10380026


### Read in DFs defining isolateID eventID mappings

In [18]:
TBP22_Reseq_IsolateID_To_GCEvents_DF = pd.read_csv(TBP22_IsolateID_To_GCEvents_TSV, sep="\t" )

TBP22_Reseq_IsolateID_To_GCEvents_DF.shape

(22, 4)

In [19]:
TBP22_Reseq_IsolateID_To_GCEvents_DF

,SampleID,EventIDs,Roles,All_Descriptions
0,TB3898,Event_001,Control,Control for Event_001
1,TB6733,Event_001,Verification,Verification for Event_001
2,TB6599,Event_003,Verification,Verification for Event_003
3,TB6977,Event_003;Event_019,Control;Verification,Control for Event_003;Verification for Event_019
4,TB3706,Event_006,Control,Control for Event_006
5,TB3305,Event_006,Verification,Verification for Event_006
6,TB7044,Event_007,Control,Control for Event_007
7,TB6755,Event_007,Verification,Verification for Event_007
8,TB6765,Event_010,Control,Control for Event_010
9,TB6552,Event_010,Verification,Verification for Event_010


### TBP22 Hybrid Complete Genome Assembly QC Stats 

In [20]:
TBP22_22CI_AsmQC_DF = pd.read_csv(TBP22_22CI_HybridAsmQCStats_TSV_PATH,
                                  sep ="\t")
TBP22_22CI_AsmQC_DF.shape

(22, 19)

In [21]:
TBP22_22CI_AsmQC_DF.head(1)

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
0,TB3706,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397096,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706,DNA621


In [22]:
TBP22_22CI_AsmQC_DF.query("SampleID == 'TB3706'")

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
0,TB3706,1,4412093,127,129,4204,2724,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397096,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Outside_Event,TB3706,DNA621


In [23]:
TBP22_22CI_AsmQC_DF.query("SampleID == 'TB3305'")

,SampleID,numContigs_Complete,circContig_Length,circContig_Cov,Flye_EstimatedCov,Flye_ReadLen_N50,Flye_ReadLen_N90,Lineage_Asm,Lineage_AsmPP,PrimaryLineage_Asm,Dataset_Tag,SR_SRA_RunAcc,SeqReason,PB_SeqRunName,EventID,Event_Gene(s),Event_Relationship,TBP_SampleID,TGEN_SampleID
1,TB3305,1,4416251,161,174,4212,2712,lineage2.2.1,lineage2.2.1,lineage2,TBPortals2022,SRR10397175,ReseqToVerfGCE,P7529,Event_006,"Rv0979c,rpmF,PE_PGRS18",Within_Event,TB3305,DNA594


##### Create dictionaries for mapping varying sampleIDs for the same isolates

In [24]:
TGEN_To_TBP_SampleID_Dict = dict(TBP22_22CI_AsmQC_DF[['TGEN_SampleID', 'TBP_SampleID']].values)
TBP_To_TGEN_SampleID_Dict = dict(TBP22_22CI_AsmQC_DF[['TBP_SampleID', 'TGEN_SampleID']].values)
TGEN_To_SR_SRA_RunAcc_Dict = dict(TBP22_22CI_AsmQC_DF[['TGEN_SampleID', 'SR_SRA_RunAcc']].values)
TBP_To_SR_SRA_RunAcc_Dict = dict(TBP22_22CI_AsmQC_DF[['TBP_SampleID', 'SR_SRA_RunAcc']].values)


### TBP22 - Assembly + Illumina WGS + PacBio WGS File Paths

In [25]:
TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF = pd.read_csv(TBP22_Final_QCPass_22CI_Input_AsmAndReads_Paths_TSV,
                                                                  sep ="\t")

TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF.shape

(22, 8)

In [26]:
TBP22_22CI_EventVerf_AsmAndRead_PATHS_V1_DF.head(1)

,SampleID,TGEN_SampleID,SRA_RunAcc_SR,Dataset_Tag,SeqReason,Illumina_PE_FQs_PATH,PacBio_FQ_PATH,HybridAsm_FA_PATH
0,TB6733,DNA0428,SRR10379945,TBPortals_2022_PassQC,PutativeRecomb,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...,/n/data1/hms/dbmi/farhat/mm774/DownloadedData/...


# Parse `Mtb151-WGA` Gubbins Results (Main Analysis)

### Define dictionary of file paths for Gubbins analysis

In [27]:
AnalysisName = "240923.WGA151CI.V8"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8"

Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"


WGA151_Gubbins_V1_OutputDir = f"{Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

WGA151_Gubbins_OutPrefix = "Gubbins"

WGA151_Gubbins_FullPrefix_PATH  = f"{WGA151_Gubbins_V1_OutputDir}/{WGA151_Gubbins_OutPrefix}"

WGA151_Gubbins_FilePath_Dict = {}

WGA151_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]        = f"{WGA151_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
WGA151_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]   = f"{WGA151_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
WGA151_Gubbins_FilePath_Dict["BranchStats_CSV"]              = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
WGA151_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]  = f"{WGA151_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]      = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]     = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]      = f"{WGA151_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]          = f"{WGA151_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"
WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]           = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
WGA151_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]    = f"{WGA151_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 




### Parse table of all putative recomb events detected in `WGA-151-GCE` 

In [28]:
import ast


In [29]:
print( WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]) 

/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb_WGA_Analysis_V8/240923.WGA151CI.V8/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh/Gubbins.recombination_predictions.Anno.tsv


In [30]:
# Parse annotated events TSV
WGA151_GRE_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"], sep = "\t")

# Convert the string column to a list of strings
WGA151_GRE_DF['taxa_List'] = WGA151_GRE_DF['taxa_List'].apply(ast.literal_eval)

WGA151_GRE_DF["LenOfTaxaList"] = WGA151_GRE_DF["taxa_List"].apply(len)

WGA151_GRE_DF.shape

(324, 28)

In [31]:
WGA151_GRE_DF.head(1)

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,EventLen,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,LenOfTaxaList
0,NC_000962.3,GUBBINS,CDS,103600,104478,0.0,.,Node_148,Node_133,1737.432787,10,"[mada_2-31, mada_1-41, MT_0080, mada_102, TB33...",103599,104038.5,NaN,"Rv0093c,Rv0094c","Rv0093c,Rv0094c",False,False,True,False,Event_001,879,2,1,1,1,130


In [32]:
WGA151_GRE_DF.query("HHR_Count >= 2 ").shape

(1, 28)

In [33]:
WGA151_GRE_DF.query("HHR_Count >= 2 ")

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,EventLen,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,LenOfTaxaList
219,NC_000962.3,GUBBINS,CDS,2866607,2867756,0.0,.,Node_132,Node_131,1451.162781,10,"[R21893, R30420, R32929, R26778, R23146, R2898...",2866606,2867181.0,lineage2,"lppA,lppB","Rv2543,Rv2544",False,False,False,True,Event_220,1150,2,1,2,1,61


### Parse `WGA151` - Gubbins ASR SNP DFs

In [34]:
WGA151_GubASR_SNPs_All_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"], sep = "\t")
WGA151_GubASR_SNPs_All_DF.shape

(26508, 7)

In [35]:
WGA151_GubASR_SNPs_EventOnly_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"], sep = "\t")
WGA151_GubASR_SNPs_EventOnly_DF.shape

(2916, 7)

### Parse Gubbins recombination events counted over "1 kb windows", "Gene-level"

In [36]:
WGA151_GRE_GeneLevel_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"], sep = "\t")

WGA151_GRE_GeneLevel_Atleast1_DF = WGA151_GRE_GeneLevel_DF.query("pGCE_Count > 0")
WGA151_GRE_GeneLevel_Atleast1_DF.shape

(76, 14)

In [37]:
WGA151_GRE_PerHHRStats_DF = pd.read_csv(WGA151_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"] , sep = "\t")
WGA151_GRE_PerHHRStats_DF.shape

(197, 19)

In [38]:
WGA151_GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(54, 19)

In [39]:
WGA151_GRE_PerHHRStats_DF["pGCE_Count"].sum()

296

In [40]:
WGA151_GRE_PerHHRStats_DF.query("pGCE_Count > 0").shape

(54, 19)

In [41]:
WGA151_GRE_PerHHRStats_DF.head(1)

,HmRegion_Num,Chr,Start,End,Num_Ovrlap_Hm_Regions,Center,Length,num_HomologRegions_NonOvrlap,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Overlap_TE,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,pGCE_Count,CenterOfRegion
0,0,NC_000962.3,80184,80523,1,80353.5,339,1,1,1,Rv0071,0,False,False,False,True,HmRegion_000,0,80353.5


### Parse in WGA-151CI Gene Conversion Results per High-Homology-Region

In [42]:
RecombEvent_To_HmRegion_CompDir = f"{Target_Output_Dir}/RecombEvent-To-HmRegion-Comparison-V2"

HmPairs_MappedEvents_TSV = f"{RecombEvent_To_HmRegion_CompDir}/240718.HmPair.MappedEventCounts.tsv"
HmRegions_MappedEvents_TSV = f"{RecombEvent_To_HmRegion_CompDir}/240718.HmRegions.MappedEventCounts.tsv"

WGA151_HHRs_GCEs_DonorAndEventCt_DF = pd.read_csv(HmRegions_MappedEvents_TSV, sep = "\t")
WGA151_HHRs_GCEs_DonorAndEventCt_DF["N_Events_Mapped"] = WGA151_HHRs_GCEs_DonorAndEventCt_DF["N_Events_HighQC"] 
WGA151_HHRs_GCEs_DonorAndEventCt_DF.shape

(197, 23)

In [43]:
WGA151_HHRs_GCEs_DonorAndEventCt_DF.head()  

,HmRegion_Num,Chr,Start,End,Center,Length,num_HomologRegions_NonOvrlap_MinSeqID99,num_HomologRegions_NonOvrlap_MinSeqID100,Overlap_Genes,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,HmRegionID,Norm_Donor_Count,N_Events_HighQC,N_Events_Putative,EventSkew,DonAccCt,EventSkew_Norm,FractionMapped,PR_SetID,N_Events_Mapped
0,0,NC_000962.3,80184,80523,80353.5,339,1,1,Rv0071,False,False,False,True,HmRegion_000,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_1,0
1,1,NC_000962.3,80623,82664,81643.5,2041,1,1,"Rv0072,Rv0073",False,False,False,True,HmRegion_001,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_2,0
2,2,NC_000962.3,103705,105130,104417.5,1425,2,2,"Rv0094c,Rv0095c",False,False,True,False,HmRegion_002,10.5,19,31,8.5,29.5,0.288136,0.612903,PR_Set_3,19
3,3,NC_000962.3,149571,149808,149689.5,237,1,1,PE_PGRS2,False,True,False,False,HmRegion_003,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_4,0
4,4,NC_000962.3,177203,177447,177325.0,244,1,1,NaN,False,False,False,True,HmRegion_004,0.0,0,0,0.0,0.0,NaN,NaN,PR_Set_5,0


# Parse `TGEN-937-SR` Gubbins Results

### Define directory paths to `TGEN-937-SR` Gubbins results

In [44]:
# Define varaint calling pipeline output directories

TGen1K_SRWGS_OutputDir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv/250721.TGEN1K.GCAnalysis"


# Define path to TGEN01K Gubbins output (SR-WGS based predictions)
TGen1K_SRWGS_Gubbins_OutDir = f"{TGen1K_SRWGS_OutputDir}/Gubbins_Analysis_V1"

Gubbins_V1_OutputDir = TGen1K_SRWGS_Gubbins_OutDir


TGENSR_Gubbins_OutPrefix = "Tgen_937CI.Gubbins.FromSNVs.V1"

TGENSR_Gubbins_FullPrefix_PATH  = f"{TGen1K_SRWGS_Gubbins_OutDir}/{TGENSR_Gubbins_OutPrefix}"

TGENSR_Gubbins_FilePath_Dict = {}

TGENSR_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]        = f"{TGENSR_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
TGENSR_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]   = f"{TGENSR_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
TGENSR_Gubbins_FilePath_Dict["BranchStats_CSV"]              = f"{TGENSR_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
TGENSR_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]  = f"{TGENSR_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
TGENSR_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]      = f"{TGENSR_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
TGENSR_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]     = f"{TGENSR_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
TGENSR_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]      = f"{TGENSR_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

TGENSR_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]          = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"
TGENSR_Gubbins_FilePath_Dict["RecombPred_CompToWGA151Events_TSV"] = f"{TGENSR_Gubbins_FullPrefix_PATH}.recombination_predictions.AnnoByOvrlapWiPrimAnalysis.V1.tsv"

TGENSR_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]           = f"{TGENSR_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
TGENSR_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]    = f"{TGENSR_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 


### Parse `TGENSR` recomb events TSVs

In [45]:
# Parse annotated events TSV
TGENSR_GRE_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"], sep = "\t")
TGENSR_GRE_DF.shape

(27, 28)

In [46]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["RecombPred_CompToWGA151Events_TSV"], sep = "\t")
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.shape

(27, 35)

In [47]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF

,seqname,source,feature,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,HmAln_Count,HmAln_Ovrlap,HHR_Count,HHR_Ovrlap,Overlap_HHRs,LenOfTaxaList,Ovrlap_HRR_Wi_pGCE_InWGA151,Ovrlap_HRR_Wi_mGCE_InWGA151,Ovrlap_Any_pGCE_InWGA151,Reseq_With_PacBioHifi,Reseq_VerficationIsolate,Reseq_ControlIsolate
0,NC_000962.3,GUBBINS,CDS,333120,333212,0.0,.,internal_985,internal_986,158.013591,8,"['SRR10379945', 'SRR7516364', 'SRR10380193', '...",333119,333165.5,93,lineage4,"vapC25,vapB25","Rv0277c,Rv0277A",False,False,False,True,Event_001,1,1,1,1,HmRegion_009,34,1,1,0,True,TB6733,TB3898
1,NC_000962.3,GUBBINS,CDS,473800,473971,0.0,.,internal_1262,SRR6807674,70.598206,6,['SRR6807674'],473799,473885.0,172,lineage4,Rv0393,Rv0393,False,False,True,False,Event_002,1,1,1,1,HmRegion_018,1,1,1,2,False,NaN,NaN
2,NC_000962.3,GUBBINS,CDS,841087,841448,0.0,.,internal_1061,internal_1062,1023.196321,14,"['SRR10380227', 'SRR10379958']",841086,841267.0,362,lineage4,"vapB31,vapC31","Rv0748,Rv0749",False,False,False,True,Event_003,1,1,1,1,HmRegion_031,2,1,1,2,True,TB6599,TB6977
3,NC_000962.3,GUBBINS,CDS,842026,842065,0.0,.,internal_941,SRR6807728,1309.669305,9,['SRR6807728'],842025,842045.0,40,lineage4,Rv0750,Rv0750,False,False,False,True,Event_004,1,1,1,1,HmRegion_032,1,1,1,3,False,NaN,NaN
4,NC_000962.3,GUBBINS,CDS,842051,842111,0.0,.,internal_1177,internal_1181,374.835261,7,"['SRR6807675', 'SRR10379983', 'SRR6807700', 'S...",842050,842080.5,61,lineage4,Rv0750,Rv0750,False,False,False,True,Event_005,1,1,1,1,HmRegion_032,145,1,1,3,False,NaN,NaN
5,NC_000962.3,GUBBINS,CDS,1094538,1095317,0.0,.,internal_1505,SRR10397175,92.819244,7,['SRR10397175'],1094537,1094927.0,780,lineage2,"Rv0979c,rpmF,PE_PGRS18","Rv0979c,Rv0979A,Rv0980c",False,True,False,False,Event_006,3,1,2,1,"HmRegion_039,HmRegion_040",1,2,2,2,True,TB3305,TB3706
6,NC_000962.3,GUBBINS,CDS,1276321,1276588,0.0,.,internal_943,internal_946,69.803852,5,"['SRR10380134', 'SRR10380230', 'SRR10379994', ...",1276320,1276454.0,268,lineage4,Rv1148c,Rv1148c,False,False,True,False,Event_007,1,1,1,1,HmRegion_049,21,1,1,4,True,TB6755,TB7044
7,NC_000962.3,GUBBINS,CDS,1339399,1339905,0.0,.,internal_1166,internal_1167,600.958317,5,"['SRR6807683', 'SRR10380192', 'SRR7516429', 'S...",1339398,1339651.5,507,lineage4,PPE18,Rv1196,False,True,False,False,Event_008,2,1,1,1,HmRegion_052,159,1,1,6,False,NaN,NaN
8,NC_000962.3,GUBBINS,CDS,1339894,1340208,0.0,.,internal_941,SRR6807728,1282.018277,8,['SRR6807728'],1339893,1340050.5,315,lineage4,PPE18,Rv1196,False,True,False,False,Event_009,2,1,1,1,HmRegion_052,1,1,1,3,False,NaN,NaN
9,NC_000962.3,GUBBINS,CDS,1340052,1341254,0.0,.,internal_953,SRR10380108,397.130081,5,['SRR10380108'],1340051,1340652.5,1203,lineage4,"PPE18,esxK,esxL","Rv1196,Rv1197,Rv1198",True,True,False,False,Event_010,6,1,2,1,"HmRegion_052,HmRegion_053",1,2,2,12,True,TB6552,TB6765


In [48]:
#TGENSR_GRE_DF.head(2)

In [49]:
#TGENSR_GRE_AnnoByWGA151Ovrlap_DF.head(2)

### Parse `TGENSR` - Gubbins ASR SNP DFs

In [50]:
TGENSR_GubASR_SNPs_All_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"], sep = "\t")
TGENSR_GubASR_SNPs_All_DF.shape

(37125, 7)

In [51]:
TGENSR_GubASR_SNPs_EventOnly_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"], sep = "\t")
TGENSR_GubASR_SNPs_EventOnly_DF.shape

(177, 7)

### Parse Gubbins recombination events counted over "1 kb windows", "Gene-level", "Merged Homologous Regions"

In [52]:
GCE_Per1kb_Stats_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"], sep = "\t")
GCE_Per1kb_Stats_DF["CenterOfRegion"] = ((GCE_Per1kb_Stats_DF["Start"] + GCE_Per1kb_Stats_DF["End"]) / 2)
GCE_Per1kb_Stats_DF.shape

(4412, 7)

In [53]:
GCE_PerGene_Stats_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"], sep = "\t")
GCE_PerGene_Stats_DF["CenterOfRegion"] = ((GCE_PerGene_Stats_DF["Start"] + GCE_PerGene_Stats_DF["End"]) / 2)
GCE_PerGene_Stats_DF.shape

(4079, 14)

In [54]:
GCE_PerGene_Stats_Atleast1_DF = GCE_PerGene_Stats_DF.query("pGCE_Count > 0")
GCE_PerGene_Stats_Atleast1_DF.shape

(25, 14)

In [55]:
TGEN_GCE_PerHHR_Stats_DF = pd.read_csv(TGENSR_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"], sep = "\t")

PerHHR_SelectCol = ["HmRegionID", "Overlap_Genes", "Chr", "Start", "End", "Center", "Length", "pGCE_Count" ] 

TGEN_GCE_PerHHR_Stats_V2_DF = TGEN_GCE_PerHHR_Stats_DF[PerHHR_SelectCol]

TGEN_GCE_PerHHR_Stats_V2_DF.shape

(197, 8)

In [56]:
TGEN_GCE_PerHHR_Stats_V2_DF.head()

,HmRegionID,Overlap_Genes,Chr,Start,End,Center,Length,pGCE_Count
0,HmRegion_000,Rv0071,NC_000962.3,80184,80523,80353.5,339,0
1,HmRegion_001,"Rv0072,Rv0073",NC_000962.3,80623,82664,81643.5,2041,0
2,HmRegion_002,"Rv0094c,Rv0095c",NC_000962.3,103705,105130,104417.5,1425,0
3,HmRegion_003,PE_PGRS2,NC_000962.3,149571,149808,149689.5,237,0
4,HmRegion_004,NaN,NC_000962.3,177203,177447,177325.0,244,0


In [57]:
TGEN_GCE_PerHHR_Stats_V2_DF.shape

(197, 8)

# Parse `TBP-22CI-LR` Gubbins Results

### Define directory paths to `TBP-22CI-LR` Gubbins results

In [58]:
# Define varaint calling pipeline output directories

AnalysisName = "250801.TBP22_22CI.V1"

Main_Project_Dir = "/n/data1/hms/dbmi/farhat/mm774/Projects/Mtb-GeneConv"

TBP22_Target_Output_Dir = f"{Main_Project_Dir}/{AnalysisName}"

# Define path to TGEN01K Gubbins output (SR-WGS based predictions)
TBP22_Gubbins_OutDir = f"{TBP22_Target_Output_Dir}/Gubbins_v321_ExtSearch_SW_MS4_mpileup_SNVs_10AmbThresh"

# Gubbins_V1_OutputDir = TGen1K_SRWGS_Gubbins_OutDir

TBP22_Gubbins_OutPrefix = "Gubbins"

TBP22_Gubbins_FullPrefix_PATH  = f"{TBP22_Gubbins_OutDir}/{TBP22_Gubbins_OutPrefix}"

TBP22_Gubbins_FilePath_Dict = {}

TBP22_Gubbins_FilePath_Dict["NodeLabelledTree_PATH"]        = f"{TBP22_Gubbins_FullPrefix_PATH}.node_labelled.final_tree.tre"
TBP22_Gubbins_FilePath_Dict["NodeToPriLineage_Dict_JSON"]   = f"{TBP22_Gubbins_FullPrefix_PATH}.NodeToPrimaryLineage.json"
TBP22_Gubbins_FilePath_Dict["BranchStats_CSV"]              = f"{TBP22_Gubbins_FullPrefix_PATH}.per_branch_statistics.csv"
TBP22_Gubbins_FilePath_Dict["BranchStats_WithLineage_CSV"]  = f"{TBP22_Gubbins_FullPrefix_PATH}.per_branch_statistics.WithLineagePerNode.tsv"
TBP22_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"]      = f"{TBP22_Gubbins_FullPrefix_PATH}.H37Rv.EventsPer1kb.tsv"
TBP22_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"]     = f"{TBP22_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerGene.tsv"
TBP22_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"]      = f"{TBP22_Gubbins_FullPrefix_PATH}.H37Rv.EventsPerMergedHomologousRegion.tsv"

TBP22_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"]          = f"{TBP22_Gubbins_FullPrefix_PATH}.recombination_predictions.Anno.tsv"

TBP22_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"]           = f"{TBP22_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.All.tsv"  
TBP22_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"]    = f"{TBP22_Gubbins_FullPrefix_PATH}.branch_base_reconstruction.AnnoByEvent.EventSNPsOnly.tsv" 


### Parse `TBP22` recomb events TSVs

In [59]:
# Parse annotated events TSV
TBP22_GRE_DF = pd.read_csv(TBP22_Gubbins_FilePath_Dict["RecombPred_Anno_TSV"], sep = "\t")
TBP22_GRE_DF.shape

(84, 26)

In [60]:
TBP22_GRE_DF.head()

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs
0,NC_000962.3,103756,103896,0.0,.,Node_8,TB6977,876.132298,4,['TB6977'],103755,103825.5,141,lineage4,Rv0094c,Rv0094c,False,False,True,False,Event_001,2,1,0,0,PR_HmRegion_002
1,NC_000962.3,103776,105045,0.0,.,Node_20,Node_3,1661.496090,14,"['TB6552', 'TB6755', 'TB6765']",103775,104410.0,1270,lineage4,"Rv0094c,Rv0095c","Rv0094c,Rv0095c",False,False,True,False,Event_002,2,1,0,0,PR_HmRegion_002
2,NC_000962.3,104712,104943,0.0,.,Node_7,Node_4,896.335956,23,"['TB6778', 'TB6973']",104711,104827.0,232,lineage4,Rv0095c,Rv0095c,False,False,True,False,Event_003,2,1,0,0,PR_HmRegion_002
3,NC_000962.3,104712,105021,0.0,.,Node_21,Node_20,3789.314751,24,"['TB6552', 'TB6755', 'TB6765', 'TB6778', 'TB69...",104711,104866.0,310,lineage4,Rv0095c,Rv0095c,False,False,True,False,Event_004,2,1,0,0,PR_HmRegion_002
4,NC_000962.3,104767,104858,0.0,.,Node_3,TB6552,469.621555,5,['TB6552'],104766,104812.0,92,lineage4,Rv0095c,Rv0095c,False,False,True,False,Event_005,2,1,0,0,PR_HmRegion_002


In [61]:
84 - 12

72

### Parse `TBP22` - Gubbins ASR SNP DFs

In [62]:
TBP22_GubASR_SNPs_All_DF = pd.read_csv(TBP22_Gubbins_FilePath_Dict["BaseAncRec_All_TSV"], sep = "\t")
TBP22_GubASR_SNPs_All_DF.shape

(5814, 7)

In [63]:
TBP22_GubASR_SNPs_EventOnly_DF = pd.read_csv(TBP22_Gubbins_FilePath_Dict["BaseAncRec_EventsOnly_TSV"], sep = "\t")
TBP22_GubASR_SNPs_EventOnly_DF.shape

(966, 7)

### Parse Gubbins GC events counted over regions of varying resolutions

#### H37Rv - "1 kb windows"

In [64]:
TBP22_GCE_Per1kb_Stats_DF = pd.read_csv(TBP22_Gubbins_FilePath_Dict["EventsPer_1kb_H37Rv_TSV"], sep = "\t")
TBP22_GCE_Per1kb_Stats_DF.shape

(4412, 7)

#### H37Rv - Per Gene

In [65]:
TBP22_GCE_PerGene_Stats_DF = pd.read_csv(TBP22_Gubbins_FilePath_Dict["EventsPer_Gene_H37Rv_TSV"], sep = "\t")
TBP22_GCE_PerGene_Stats_DF.shape

(4079, 14)

In [66]:
TBP22_GCE_PerGene_Stats_Atleast1_DF = TBP22_GCE_PerGene_Stats_DF.query("pGCE_Count > 0")
TBP22_GCE_PerGene_Stats_Atleast1_DF.shape

(46, 14)

In [67]:
TBP22_GCE_PerGene_Stats_Atleast1_DF.head(1)

,Chrom,Start,End,Strand,H37rv_GeneID,Symbol,Feature,Functional_Category,Is_Pseudogene,Product,PEandPPE_Subfamily,ExcludedGroup_Category,pGCE_Count,CenterOfRegion
99,NC_000962.3,103709,104663,-,Rv0094c,Rv0094c,CDS,insertion seqs and phages,No,Conserved hypothetical protein,NaN,InsertionSeqs_And_Phages,2,104186.0


#### H37Rv - Per HHR

In [68]:
TBP22_GCE_PerHHR_Stats_DF = pd.read_csv(TBP22_Gubbins_FilePath_Dict["EventsPer_HHR_H37Rv_TSV"], sep = "\t")

PerHHR_SelectCol = ["HmRegionID", "Overlap_Genes", "Chr", "Start", "End", "Center", "Length", "pGCE_Count", "Overlap_GC_EventIDs" ] 

TBP22_GCE_PerHHR_Stats_V2_DF = TBP22_GCE_PerHHR_Stats_DF[PerHHR_SelectCol]
TBP22_GCE_PerHHR_Stats_V2_DF.shape

(200, 9)

In [69]:
TBP22_GCE_PerHHR_Stats_V2_DF.sort_values("pGCE_Count", ascending=False).head(4)

,HmRegionID,Overlap_Genes,Chr,Start,End,Center,Length,pGCE_Count,Overlap_GC_EventIDs
2,PR_HmRegion_002,"Rv0094c,Rv0095c",NC_000962.3,103705,105130,104417.5,1425,11,"Event_001,Event_002,Event_003,Event_004,Event_..."
65,PR_HmRegion_065,"PE_PGRS28,Rv1453",NC_000962.3,1636705,1639561,1638133.0,2856,5,"Event_039,Event_040,Event_041,Event_042,Event_043"
179,PR_HmRegion_179,"PPE59,Rv3430c",NC_000962.3,3846460,3847892,3847176.0,1432,4,"Event_072,Event_073,Event_074,Event_075"
64,PR_HmRegion_064,PE_PGRS27,NC_000962.3,1633310,1634790,1634050.0,1480,4,"Event_035,Event_036,Event_037,Event_038"


In [70]:
TBP22_GCE_PerHHR_Stats_DF.shape

(200, 13)

### Create DF of HHRs w/ more than 1 GC Event detected in `TGENSR` dataset

In [71]:
TBP22_GCE_PerHHR_WiOvrLapGCE_Stats_DF = TBP22_GCE_PerHHR_Stats_DF.query("pGCE_Count > 0")

TBP22_GCE_PerHHR_WiOvrLapGCE_Stats_DF.shape

(35, 13)

# Define functions for comparison of detected mutations in Gubbins ASR SNP DFs

In [72]:

def compare_Gubbins_ASR_SNP_DFs(df_a, df_b):
    """
    Compare SNPs between two DataFrames and return:
    1. Number of intersecting SNPs with matching parent and child calls
    2. DataFrame of intersecting SNPs
    3. Outer-merged DataFrame of both SNP sets

    Parameters:
    - df_a: DataFrame with columns ["Pos_1based", "Parent_Call", "Child_Call", "EventID"]
    - df_b: DataFrame with columns ["Pos_1based", "Parent_Call", "Child_Call", "EventID"]

    Returns:
    - N_Intersect_SNPs: int
    - intersect_df: pd.DataFrame
    - outer_merged_df: pd.DataFrame
    """
    # Subset relevant columns
    df_a_trim = df_a[["Pos_1based", "Parent_Call", "Child_Call", "EventID"]]
    df_b_trim = df_b[["Pos_1based", "Parent_Call", "Child_Call", "EventID"]]

    # Outer merge on position
    outer_merged_df = pd.merge(df_a_trim, df_b_trim, how='outer',
                               on='Pos_1based',
                               suffixes=("_A", "_B"))

    # Filter for matching SNPs
    intersect_df = outer_merged_df.query(
        "(Parent_Call_A == Parent_Call_B) & (Child_Call_A == Child_Call_B)"
    )

    N_Intersect_SNPs = intersect_df.shape[0]

    return N_Intersect_SNPs, intersect_df, outer_merged_df


# Part 2: Validate each `TGENSR` detected GC event
Evaluate each putative event detected in `TGENSR` dataset using the dataset of isolates resequenced w/ PacBio HiFi WGS (`TBP22`)


In [73]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF #[["EventID", "Verification_IsolateID", "Control_IsolateID"]]

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
0,Event_001,TB6733,TB3898,SRR10379945,SRR10397263
1,Event_003,TB6599,TB6977,SRR10379958,SRR10380218
2,Event_006,TB3305,TB3706,SRR10397175,SRR10397096
3,Event_007,TB6755,TB7044,SRR10379935,SRR10380192
4,Event_010,TB6552,TB6765,SRR10380108,SRR10379924
5,Event_011,TB6778,TB6973,SRR10380252,SRR10380223
6,Event_013,TB6786,TB3256,SRR10380244,SRR10397205
7,Event_019,TB6977,TB6976,SRR10380218,SRR10380219
8,Event_021,TB3572,TB6976,SRR10397163,SRR10380219
9,Event_022,TB6596,TB8073,SRR10379961,SRR10380026


In [74]:
TBP22_Reseq_IsolateID_To_GCEvents_DF

,SampleID,EventIDs,Roles,All_Descriptions
0,TB3898,Event_001,Control,Control for Event_001
1,TB6733,Event_001,Verification,Verification for Event_001
2,TB6599,Event_003,Verification,Verification for Event_003
3,TB6977,Event_003;Event_019,Control;Verification,Control for Event_003;Verification for Event_019
4,TB3706,Event_006,Control,Control for Event_006
5,TB3305,Event_006,Verification,Verification for Event_006
6,TB7044,Event_007,Control,Control for Event_007
7,TB6755,Event_007,Verification,Verification for Event_007
8,TB6765,Event_010,Control,Control for Event_010
9,TB6552,Event_010,Verification,Verification for Event_010


In [75]:
GCE_TargetColn_Set2 = ["EventID", "Overlap_Genes", "start_0based", "end_1based", "EventLen", "snp_count", #"LenOfTaxaList",
                   "Child_Node",
                   "Lineage",   
                   "Overlap_HHRs", "taxa_List"] #"HHR_Ovrlap",
                   # "Ovrlap_HRR_Wi_pGCE_InWGA151",
                   # "Reseq_With_PacBioHifi",
                   # "Reseq_VerficationIsolate", "Reseq_ControlIsolate"] 
GCE_TargetColn_Set3 = ["EventID", "Overlap_Genes", "start_0based", "end_1based", "EventLen", "snp_count", #"LenOfTaxaList",
                   "Child_Node",
                   "Lineage",   
                   "Overlap_HHRs", "Ovrlap_HRR_Wi_pGCE_InWGA151", "Reseq_With_PacBioHifi", ] #"HHR_Ovrlap",

In [76]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("Reseq_With_PacBioHifi == True")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
0,Event_001,"vapC25,vapB25",333119,333212,93,8,internal_986,lineage4,HmRegion_009,"['SRR10379945', 'SRR7516364', 'SRR10380193', '..."
2,Event_003,"vapB31,vapC31",841086,841448,362,14,internal_1062,lineage4,HmRegion_031,"['SRR10380227', 'SRR10379958']"
5,Event_006,"Rv0979c,rpmF,PE_PGRS18",1094537,1095317,780,7,SRR10397175,lineage2,"HmRegion_039,HmRegion_040",['SRR10397175']
6,Event_007,Rv1148c,1276320,1276588,268,5,internal_946,lineage4,HmRegion_049,"['SRR10380134', 'SRR10380230', 'SRR10379994', ..."
9,Event_010,"PPE18,esxK,esxL",1340051,1341254,1203,5,SRR10380108,lineage4,"HmRegion_052,HmRegion_053",['SRR10380108']
10,Event_011,PPE18,1340388,1340407,19,7,internal_1140,lineage4,HmRegion_052,"['SRR10380223', 'SRR10380252', 'SRR10380251']"
12,Event_013,PPE19,1533015,1533625,610,6,internal_1039,lineage4,HmRegion_059,"['SRR10380152', 'SRR10379968', 'SRR10380088', ..."
18,Event_019,PPE46,3377270,3377320,50,8,internal_1061,lineage4,HmRegion_152,"['SRR10380227', 'SRR10379958', 'SRR10380218', ..."
20,Event_021,Rv3108,3477265,3477370,105,5,SRR10397163,lineage4,NaN,['SRR10397163']
21,Event_022,PPE56,3765796,3766860,1064,7,internal_1023,lineage4,HmRegion_170,"['SRR10380179', 'SRR10380112', 'SRR10379961', ..."


In [77]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query("Reseq_With_PacBioHifi == True")[GCE_TargetColn_Set3]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,Ovrlap_HRR_Wi_pGCE_InWGA151,Reseq_With_PacBioHifi
0,Event_001,"vapC25,vapB25",333119,333212,93,8,internal_986,lineage4,HmRegion_009,1,True
2,Event_003,"vapB31,vapC31",841086,841448,362,14,internal_1062,lineage4,HmRegion_031,1,True
5,Event_006,"Rv0979c,rpmF,PE_PGRS18",1094537,1095317,780,7,SRR10397175,lineage2,"HmRegion_039,HmRegion_040",2,True
6,Event_007,Rv1148c,1276320,1276588,268,5,internal_946,lineage4,HmRegion_049,1,True
9,Event_010,"PPE18,esxK,esxL",1340051,1341254,1203,5,SRR10380108,lineage4,"HmRegion_052,HmRegion_053",2,True
10,Event_011,PPE18,1340388,1340407,19,7,internal_1140,lineage4,HmRegion_052,1,True
12,Event_013,PPE19,1533015,1533625,610,6,internal_1039,lineage4,HmRegion_059,1,True
18,Event_019,PPE46,3377270,3377320,50,8,internal_1061,lineage4,HmRegion_152,1,True
20,Event_021,Rv3108,3477265,3477370,105,5,SRR10397163,lineage4,NaN,0,True
21,Event_022,PPE56,3765796,3766860,1064,7,internal_1023,lineage4,HmRegion_170,1,True


### Gameplan: 
1) Go over each candidate event and check if there is the corresponding isolate contains an overlapping recombination event
2) Evaluate coordinate and variant overlap between each event
   - Is the putative event supported by LR analysis?
3) Summarize results


## A - `Event_001` - `vapC25,vapB25` - **VERFIED!**

In [78]:
i_Tar_EventID = 'Event_001'

In [79]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
0,Event_001,"vapC25,vapB25",333119,333212,93,8,internal_986,lineage4,HmRegion_009,"['SRR10379945', 'SRR7516364', 'SRR10380193', '..."


In [80]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
0,Event_001,TB6733,TB3898,SRR10379945,SRR10397263


In [81]:
TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

,SampleID,EventIDs,Roles,All_Descriptions
0,TB3898,Event_001,Control,Control for Event_001
1,TB6733,Event_001,Verification,Verification for Event_001


#### Look at events in VERFICATION isolate

In [82]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6733")].shape

(13, 26)

In [83]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6733") & TBP22_GRE_DF["Overlap_Genes"].str.contains("vapC25")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
11,Event_012,"vapC25,vapB25",333119,333212,93,19,Node_14,lineage4,PR_HmRegion_009,"['TB6733', 'TB6846']"


#### Look at events in CONTROL isolate

In [84]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3898")].shape

(12, 26)

In [85]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3898") & TBP22_GRE_DF["Overlap_Genes"].str.contains("vapC25")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [86]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_001' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_012' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 8 
B: 19


In [87]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (19, 7)
Overlap set of SNPs DF: (8, 7)
i_N_Ovrlap_SNPs: 8


In [88]:
i_N_Ovrlap_SNPs

8

In [89]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.42105263157894735

In [90]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

##### Prototyping for function dev

In [91]:
TBP22_GubASR_SNPs_EventOnly_DF.head(1)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
0,1094228,Node_1,TB3305,T,G,TB3305,Event_019


In [92]:
TGENSR_GubASR_SNPs_EventOnly_DF.head(1)

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
0,3765797,internal_1022,internal_1023,G,C,SRR10380179 SRR10380112 SRR10379961 SRR1038...,Event_022


In [93]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_001' ")
i_TGENSR_Event_SNPs_DF.shape

(8, 7)

In [94]:
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_012' ")
i_TBP22_Event_SNPs_DF.shape

(19, 7)

In [95]:
i_TBP22_Event_SNPs_DF.head()

,Pos_1based,Parent_Node,Child_Node,Parent_Call,Child_Call,taxa_List,EventID
215,333120,Node_15,Node_14,A,G,TB6733 TB6846,Event_012
216,333127,Node_15,Node_14,T,G,TB6733 TB6846,Event_012
217,333128,Node_15,Node_14,G,A,TB6733 TB6846,Event_012
218,333137,Node_15,Node_14,C,T,TB6733 TB6846,Event_012
219,333141,Node_15,Node_14,T,G,TB6733 TB6846,Event_012


In [96]:
i_A_SNPs_DF = i_TGENSR_Event_SNPs_DF
i_B_SNPs_DF = i_TBP22_Event_SNPs_DF

i_A_SNPs_Trim_DF = i_A_SNPs_DF[["Pos_1based", "Parent_Call", "Child_Call", "EventID"]]
i_B_SNPs_Trim_DF = i_B_SNPs_DF[["Pos_1based", "Parent_Call", "Child_Call", "EventID"]]

# Outer Merging the two dataframes on there position
i_AandB_OMerge_DF = pd.merge(i_A_SNPs_DF, i_B_SNPs_DF, how='outer',
                             on='Pos_1based', 
                             suffixes = ("_A", "_B"))
print(i_AandB_OMerge_DF.shape)

# Get inersection of the two input SNP sets
i_AandB_Intersect_DF = i_AandB_OMerge_DF.query("( Parent_Call_A == Parent_Call_B) & (Child_Call_A == Child_Call_B)")
print(i_AandB_Intersect_DF.shape)

N_Intersect_SNPs = i_AandB_Intersect_DF.shape[0]

# I want to return N_Intersect_SNPs, i_AandB_Intersect_DF, and i_AandB_OMerge_DF as part of the function


(19, 13)
(8, 13)


In [97]:
i_AandB_Intersect_DF = i_AandB_OMerge_DF.query("( Parent_Call_A == Parent_Call_B) & (Child_Call_A == Child_Call_B)")
i_AandB_Intersect_DF.shape

(8, 13)

In [98]:
i_AandB_Intersect_DF

,Pos_1based,Parent_Node_A,Child_Node_A,Parent_Call_A,Child_Call_A,taxa_List_A,EventID_A,Parent_Node_B,Child_Node_B,Parent_Call_B,Child_Call_B,taxa_List_B,EventID_B
0,333120,internal_985,internal_986,A,G,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,A,G,TB6733 TB6846,Event_012
1,333127,internal_985,internal_986,T,G,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,T,G,TB6733 TB6846,Event_012
2,333128,internal_985,internal_986,G,A,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,G,A,TB6733 TB6846,Event_012
14,333169,internal_985,internal_986,C,G,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,C,G,TB6733 TB6846,Event_012
15,333184,internal_985,internal_986,A,G,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,A,G,TB6733 TB6846,Event_012
16,333209,internal_985,internal_986,A,G,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,A,G,TB6733 TB6846,Event_012
17,333211,internal_985,internal_986,C,T,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,C,T,TB6733 TB6846,Event_012
18,333212,internal_985,internal_986,G,C,SRR10379945 SRR7516364 SRR10380193 SRR1037...,Event_001,Node_15,Node_14,G,C,TB6733 TB6846,Event_012


In [99]:
N_Intersect_SNPs = i_AandB_Intersect_DF.shape[0]
N_Intersect_SNPs

8

## B - `Event_003` - `vapB31,vapC31` - **VERFIED!**

In [100]:
i_Tar_EventID = 'Event_003'

In [101]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
2,Event_003,"vapB31,vapC31",841086,841448,362,14,internal_1062,lineage4,HmRegion_031,"['SRR10380227', 'SRR10379958']"


In [102]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
1,Event_003,TB6599,TB6977,SRR10379958,SRR10380218


In [103]:
TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

,SampleID,EventIDs,Roles,All_Descriptions
2,TB6599,Event_003,Verification,Verification for Event_003
3,TB6977,Event_003;Event_019,Control;Verification,Control for Event_003;Verification for Event_019


#### Look at events in VERFICATION isolate

In [104]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6599")].shape

(19, 26)

In [105]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6599") & TBP22_GRE_DF["Overlap_Genes"].str.contains("vapB31")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
16,Event_017,"vapB31,vapC31",841070,841448,378,21,TB6599,lineage4,PR_HmRegion_032,['TB6599']


#### Look at events in CONTROL isolate

In [106]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6977")].shape

(16, 26)

In [107]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6977") & TBP22_GRE_DF["Overlap_Genes"].str.contains("vapB31")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [108]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_003' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_017' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 14 
B: 21


In [109]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (21, 7)
Overlap set of SNPs DF: (14, 7)
i_N_Ovrlap_SNPs: 14


In [110]:
i_N_Ovrlap_SNPs

14

In [111]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.6666666666666666

In [112]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

## C - `Event_006` - `PE_PGRS17` - **VERFIED!** (True event is in `PE-PGRS17` not in `PE-PGRS18`)

In [113]:
i_Tar_EventID = 'Event_006'

In [114]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
5,Event_006,"Rv0979c,rpmF,PE_PGRS18",1094537,1095317,780,7,SRR10397175,lineage2,"HmRegion_039,HmRegion_040",['SRR10397175']


In [115]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
2,Event_006,TB3305,TB3706,SRR10397175,SRR10397096


In [116]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [117]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3305")].shape

(1, 26)

In [118]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3305")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
18,Event_019,PE_PGRS17,1094227,1094576,349,7,TB3305,lineage2,PR_HmRegion_040,['TB3305']


In [119]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3305") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PE_PGRS17")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
18,Event_019,PE_PGRS17,1094227,1094576,349,7,TB3305,lineage2,PR_HmRegion_040,['TB3305']


#### Look at events in CONTROL isolate

In [120]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3706")].shape

(0, 26)

In [121]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3706") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PE_PGRS18")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [122]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_006' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_019' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 7 
B: 7


In [123]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (9, 7)
Overlap set of SNPs DF: (5, 7)
i_N_Ovrlap_SNPs: 5


In [124]:
i_N_Ovrlap_SNPs

5

In [125]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.7142857142857143

In [126]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

0.7142857142857143

In [127]:
#i_Merge_SNPs_DF

## D - `Event_007` - `Rv1148c` - **VERFIED!**

In [128]:
i_Tar_EventID = 'Event_007'

In [129]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
6,Event_007,Rv1148c,1276320,1276588,268,5,internal_946,lineage4,HmRegion_049,"['SRR10380134', 'SRR10380230', 'SRR10379994', ..."


In [130]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
3,Event_007,TB6755,TB7044,SRR10379935,SRR10380192


In [131]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [132]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6755")].shape

(17, 26)

In [133]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6755") & TBP22_GRE_DF["Overlap_Genes"].str.contains("Rv1148c")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
25,Event_026,Rv1148c,1276320,1276885,565,42,Node_3,lineage4,PR_HmRegion_050,"['TB6552', 'TB6755', 'TB6765']"
26,Event_027,Rv1148c,1276841,1276885,44,11,Node_20,lineage4,PR_HmRegion_050,"['TB6552', 'TB6755', 'TB6765', 'TB6778', 'TB69..."


#### Look at events in CONTROL isolate

In [134]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB7044")].shape

(14, 26)

In [135]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB7044") & TBP22_GRE_DF["Overlap_Genes"].str.contains("Rv1148c")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
26,Event_027,Rv1148c,1276841,1276885,44,11,Node_20,lineage4,PR_HmRegion_050,"['TB6552', 'TB6755', 'TB6765', 'TB6778', 'TB69..."


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [136]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_007' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_026' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 5 
B: 42


In [137]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (42, 7)
Overlap set of SNPs DF: (5, 7)
i_N_Ovrlap_SNPs: 5


In [138]:
i_N_Ovrlap_SNPs

5

In [139]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.11904761904761904

In [140]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

## E - `Event_010` - `esxK,esxL` - **VERFIED!** (Does not truly occur in `PPE18`)

In [141]:
i_Tar_EventID = 'Event_010'

In [142]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
9,Event_010,"PPE18,esxK,esxL",1340051,1341254,1203,5,SRR10380108,lineage4,"HmRegion_052,HmRegion_053",['SRR10380108']


In [143]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
4,Event_010,TB6552,TB6765,SRR10380108,SRR10379924


In [144]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [145]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6552")].shape

(17, 26)

In [146]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6552") & TBP22_GRE_DF["Overlap_Genes"].str.contains("esxK")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
29,Event_030,"esxK,esxL",1340577,1341254,677,13,TB6552,lineage4,PR_HmRegion_054,['TB6552']


#### Look at events in CONTROL isolate

In [147]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6765")].shape

(16, 26)

In [148]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6765") & TBP22_GRE_DF["Overlap_Genes"].str.contains("esxK")]

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [149]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_010' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_030' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 5 
B: 13


In [150]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (15, 7)
Overlap set of SNPs DF: (3, 7)
i_N_Ovrlap_SNPs: 3


In [151]:
i_N_Ovrlap_SNPs

3

In [152]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.23076923076923078

In [153]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

0.6

In [154]:
i_Merge_SNPs_DF

,Pos_1based,Parent_Call_A,Child_Call_A,EventID_A,Parent_Call_B,Child_Call_B,EventID_B
0,1340052,G,T,Event_010,NaN,NaN,NaN
1,1340127,A,G,Event_010,NaN,NaN,NaN
2,1340578,T,C,Event_010,T,C,Event_030
3,1340664,NaN,NaN,NaN,C,A,Event_030
4,1340665,NaN,NaN,NaN,T,A,Event_030
5,1340830,NaN,NaN,NaN,G,A,Event_030
6,1340916,NaN,NaN,NaN,G,A,Event_030
7,1340996,NaN,NaN,NaN,A,G,Event_030
8,1341005,NaN,NaN,NaN,G,C,Event_030
9,1341023,NaN,NaN,NaN,G,A,Event_030


## F - `Event_011` - `PPE18`

In [155]:
i_Tar_EventID = 'Event_011'

In [156]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
10,Event_011,PPE18,1340388,1340407,19,7,internal_1140,lineage4,HmRegion_052,"['SRR10380223', 'SRR10380252', 'SRR10380251']"


In [157]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
5,Event_011,TB6778,TB6973,SRR10380252,SRR10380223


In [158]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [159]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6778")].shape

(17, 26)

In [160]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6778") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE18")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
28,Event_029,PPE18,1340388,1340407,19,8,TB6778,lineage4,PR_HmRegion_053,['TB6778']


#### Look at events in CONTROL isolate

In [161]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6973")].shape

(16, 26)

In [162]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6973") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE18")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [163]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_011' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_029' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 7 
B: 8


In [164]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (8, 7)
Overlap set of SNPs DF: (7, 7)
i_N_Ovrlap_SNPs: 7


In [165]:
i_N_Ovrlap_SNPs

7

In [166]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.875

In [167]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

In [168]:
i_Merge_SNPs_DF

,Pos_1based,Parent_Call_A,Child_Call_A,EventID_A,Parent_Call_B,Child_Call_B,EventID_B
0,1340389,G,C,Event_011,G,C,Event_029
1,1340390,G,C,Event_011,G,C,Event_029
2,1340394,G,C,Event_011,G,C,Event_029
3,1340395,NaN,NaN,NaN,A,C,Event_029
4,1340397,G,C,Event_011,G,C,Event_029
5,1340398,G,C,Event_011,G,C,Event_029
6,1340404,G,A,Event_011,G,A,Event_029
7,1340407,G,C,Event_011,G,C,Event_029


## G - `Event_013` - `PPE19`

In [169]:
i_Tar_EventID = 'Event_013'

In [170]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
12,Event_013,PPE19,1533015,1533625,610,6,internal_1039,lineage4,HmRegion_059,"['SRR10380152', 'SRR10379968', 'SRR10380088', ..."


In [171]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
6,Event_013,TB6786,TB3256,SRR10380244,SRR10397205


In [172]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [173]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6786")].shape

(16, 26)

In [174]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6786") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE19")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
32,Event_033,PPE19,1533015,1533625,610,9,TB6786,lineage4,PR_HmRegion_060,['TB6786']


#### Look at events in CONTROL isolate

In [175]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3256")].shape

(13, 26)

In [176]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3256") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE19")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [177]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_013' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_033' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 6 
B: 9


In [178]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (9, 7)
Overlap set of SNPs DF: (6, 7)
i_N_Ovrlap_SNPs: 6


In [179]:
i_N_Ovrlap_SNPs

6

In [180]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.6666666666666666

In [181]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

In [182]:
i_Merge_SNPs_DF

,Pos_1based,Parent_Call_A,Child_Call_A,EventID_A,Parent_Call_B,Child_Call_B,EventID_B
0,1533016,T,C,Event_013,T,C,Event_033
1,1533097,G,A,Event_013,G,A,Event_033
2,1533466,NaN,NaN,NaN,T,C,Event_033
3,1533470,NaN,NaN,NaN,G,A,Event_033
4,1533471,NaN,NaN,NaN,T,C,Event_033
5,1533546,T,G,Event_013,T,G,Event_033
6,1533547,C,A,Event_013,C,A,Event_033
7,1533550,G,C,Event_013,G,C,Event_033
8,1533625,G,A,Event_013,G,A,Event_033


## H - `Event_019` - `PPE46`

In [183]:
i_Tar_EventID = 'Event_019'

In [184]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
18,Event_019,PPE46,3377270,3377320,50,8,internal_1061,lineage4,HmRegion_152,"['SRR10380227', 'SRR10379958', 'SRR10380218', ..."


In [185]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
7,Event_019,TB6977,TB6976,SRR10380218,SRR10380219


In [186]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [187]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6977")].shape

(16, 26)

In [188]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6977") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE46")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
61,Event_062,PPE46,3377270,3377347,77,13,Node_8,lineage4,PR_HmRegion_155,"['TB6977', 'TB6599']"


#### Look at events in CONTROL isolate

In [189]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6976")].shape

(19, 26)

In [190]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6976") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE46")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [191]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_019' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_062' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 8 
B: 13


In [192]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (13, 7)
Overlap set of SNPs DF: (8, 7)
i_N_Ovrlap_SNPs: 8


In [193]:
i_N_Ovrlap_SNPs

8

In [194]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.6153846153846154

In [195]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

## I - `Event_021` - `Rv3108`

In [196]:
i_Tar_EventID = 'Event_021'

In [197]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
20,Event_021,Rv3108,3477265,3477370,105,5,SRR10397163,lineage4,NaN,['SRR10397163']


In [198]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
8,Event_021,TB3572,TB6976,SRR10397163,SRR10380219


In [199]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [200]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3572")].shape

(18, 26)

In [201]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB3572") & TBP22_GRE_DF["Overlap_Genes"].str.contains("Rv3108")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
62,Event_063,Rv3108,3477265,3477370,105,5,TB3572,lineage4,NaN,['TB3572']


#### Look at events in CONTROL isolate

In [202]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6976")].shape

(19, 26)

In [203]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6976") & TBP22_GRE_DF["Overlap_Genes"].str.contains("Rv3108")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [204]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_021' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_063' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 5 
B: 5


In [205]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (5, 7)
Overlap set of SNPs DF: (5, 7)
i_N_Ovrlap_SNPs: 5


In [206]:
i_N_Ovrlap_SNPs

5

In [207]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

1.0

In [208]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

## J - `Event_022` - `PPE56`

In [209]:
i_Tar_EventID = 'Event_022'

In [210]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
21,Event_022,PPE56,3765796,3766860,1064,7,internal_1023,lineage4,HmRegion_170,"['SRR10380179', 'SRR10380112', 'SRR10379961', ..."


In [211]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
9,Event_022,TB6596,TB8073,SRR10379961,SRR10380026


In [212]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [213]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6596")].shape

(13, 26)

In [214]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6596") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE56")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
70,Event_071,PPE56,3765796,3766860,1064,16,TB6596,lineage4,PR_HmRegion_173,['TB6596']


#### Look at events in CONTROL isolate

In [215]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB8073")].shape

(13, 26)

In [216]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB8073") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE56")]

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [217]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_022' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_071' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 7 
B: 16


In [218]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (16, 7)
Overlap set of SNPs DF: (7, 7)
i_N_Ovrlap_SNPs: 7


In [219]:
i_N_Ovrlap_SNPs

7

In [220]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.4375

In [221]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

## K - `Event_024` - `PPE59,Rv3430c`

In [222]:
i_Tar_EventID = 'Event_024'

In [223]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
23,Event_024,"PPE59,Rv3430c",3847545,3847664,119,8,internal_1264,lineage4,HmRegion_176,"['SRR10380093', 'SRR10380130']"


In [224]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
10,Event_024,TB7340,TB6807,SRR10380093,SRR10379998


#### Look at events in VERFICATION isolate

In [225]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB7340")].shape

(18, 26)

In [226]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB7340") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE59")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
71,Event_072,PPE59,3847214,3847238,24,5,Node_20,lineage4,PR_HmRegion_179,"['TB6552', 'TB6755', 'TB6765', 'TB6778', 'TB69..."
74,Event_075,"PPE59,Rv3430c",3847545,3847684,139,29,TB7340,lineage4,PR_HmRegion_179,['TB7340']


#### Look at events in CONTROL isolate

In [227]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6807")].shape

(17, 26)

In [228]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6807") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE59")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
71,Event_072,PPE59,3847214,3847238,24,5,Node_20,lineage4,PR_HmRegion_179,"['TB6552', 'TB6755', 'TB6765', 'TB6778', 'TB69..."


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [229]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_024' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_075' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 8 
B: 29


In [230]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (29, 7)
Overlap set of SNPs DF: (8, 7)
i_N_Ovrlap_SNPs: 8


In [231]:
i_N_Ovrlap_SNPs

8

In [232]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.27586206896551724

In [233]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

## L - `Event_025` - `PPE60`

In [234]:
i_Tar_EventID = 'Event_025'

In [235]:
TGENSR_GRE_AnnoByWGA151Ovrlap_DF.query(f"EventID == '{i_Tar_EventID}'")[GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
24,Event_025,PPE60,3894770,3894791,21,6,internal_995,lineage4,HmRegion_179,"['SRR10379925', 'SRR10380175', 'SRR7516342', '..."


In [236]:
TBP22_Reseq_GCEvent_To_IsolateIDs_DF.query(f"EventID == '{i_Tar_EventID}'")

,EventID,Verification_IsolateID,Control_IsolateID,Verfication_SRWGS_RunID,Control_SRWGS_RunID
11,Event_025,TB6846,TB4414,SRR10379893,SRR10430399


In [237]:
#TBP22_Reseq_IsolateID_To_GCEvents_DF[TBP22_Reseq_IsolateID_To_GCEvents_DF["EventIDs"].str.contains(i_Tar_EventID)]

#### Look at events in VERFICATION isolate

In [238]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6846")].shape

(16, 26)

In [239]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB6846") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE60")][GCE_TargetColn_Set2]

,EventID,Overlap_Genes,start_0based,end_1based,EventLen,snp_count,Child_Node,Lineage,Overlap_HHRs,taxa_List
77,Event_078,PPE60,3894508,3895403,895,17,TB6846,lineage4,PR_HmRegion_182,['TB6846']


#### Look at events in CONTROL isolate

In [240]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB4414")].shape

(12, 26)

In [241]:
TBP22_GRE_DF[TBP22_GRE_DF["taxa_List"].str.contains("TB4414") & TBP22_GRE_DF["Overlap_Genes"].str.contains("PPE60")]

,seqname,start_1based,end_1based,score,strand,Parent_Node,Child_Node,neg_log_likelihood,snp_count,taxa_List,start_0based,CenterOfRegion,EventLen,Lineage,Overlap_Genes,Overlap_Gene_RvIDs,Contains_Esx,Contains_PEPPE,Contains_REP13E12,NoOverlap_Wi_PEPPE_Esx_13E12_Genes,EventID,N_HmMapAln_PR_Ovrlap,OvrlapWi_HmMapAln_PR,N_HmMapAln_LR_Ovrlap,OvrlapWi_HmMapAln_LR,Overlap_HHRs


#### Now overlap detected SNPs (From Gubbins ASR) for the two analyses

In [242]:
i_TGENSR_Event_SNPs_DF = TGENSR_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_025' ")
i_TBP22_Event_SNPs_DF  = TBP22_GubASR_SNPs_EventOnly_DF.query("EventID == 'Event_078' ")

print("A:", i_TGENSR_Event_SNPs_DF.shape[0], "\nB:", i_TBP22_Event_SNPs_DF.shape[0])

A: 6 
B: 17


In [243]:
i_N_Ovrlap_SNPs, i_Ovrlap_SNPs_DF, i_Merge_SNPs_DF = compare_Gubbins_ASR_SNP_DFs(i_TGENSR_Event_SNPs_DF, i_TBP22_Event_SNPs_DF)
print("Merged set of SNPs DF :", i_Merge_SNPs_DF.shape)
print("Overlap set of SNPs DF:", i_Ovrlap_SNPs_DF.shape)
print("i_N_Ovrlap_SNPs:", i_N_Ovrlap_SNPs)

Merged set of SNPs DF : (17, 7)
Overlap set of SNPs DF: (6, 7)
i_N_Ovrlap_SNPs: 6


In [244]:
i_N_Ovrlap_SNPs

6

In [245]:
i_N_Ovrlap_SNPs / i_TBP22_Event_SNPs_DF.shape[0]

0.35294117647058826

In [246]:
i_N_Ovrlap_SNPs / i_TGENSR_Event_SNPs_DF.shape[0]

1.0

# Part 3: 

# Extras